# 06 · Questions and qrels

Plan section 2. Validates `data/eval/questions.jsonl` and derives `data/eval/qrels.jsonl` for both indexes from character spans:

* **unit / window level** (per index): grade 2 = the unit or window contains the whole quote span; grade 1 = another unit or window of a gold section. If no unit of the gold section holds the whole span (a quote cut by a base-unit split), the units overlapping it get grade 2 and are flagged `partial`.
* **section level** (same for both indexes): grade 2 = a gold section; grade 1 = a sibling in the same group.
* **group level**: grade 2 = the gold group.

Only the 92 answerable questions get qrels. The 8 unanswerable ones run through every retriever but are never scored.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
import json
import pandas as pd
from csrag.corpus import Corpus
from csrag import evaluate as ev

corpus = Corpus(DATA)
assert corpus.has_windows, "run notebook 04 first"
questions, problems = ev.load_questions(DATA / "eval" / "questions.jsonl", corpus)
for k, v in problems.items():
    print(f"{k:30} {len(v):3}  {v[:5]}")
assert not any(v for k, v in problems.items() if k != "quote_ambiguous"), "question validation failed"
print(len(questions), "questions;", sum(q["answerable"] for q in questions), "answerable")

## Distribution

In [ ]:
qf = pd.DataFrame([{k: q.get(k) for k in ("question_id", "question_type", "engine_type", "difficulty", "gold_kind",
                                          "mentions_section_id", "answerable")} for q in questions])
def length_bucket(q):
    if not q["answerable"]:
        return "unanswerable"
    n = corpus.n_tokens(corpus.sections[q["_gold"][0]["section_id"]]["body"])
    return "≤500" if n <= 500 else ("501–1500" if n <= 1500 else ">1500")
qf["length_bucket"] = [length_bucket(q) for q in questions]
for col in ("question_type", "engine_type", "difficulty", "gold_kind", "mentions_section_id", "length_bucket"):
    display(qf[qf["answerable"]][col].value_counts(dropna=False).rename(col).to_frame().T)

## Qrels for both indexes

In [ ]:
qrels = ev.build_qrels(questions, corpus)
check = ev.check_qrels(qrels, questions, corpus)
for k, v in check.items():
    print(f"{k:22} {len(v):3}  {v[:5]}")
assert not check["orphan_ids"] and not check["missing"], "qrels have orphans or gaps"
ev.write_qrels(qrels, DATA / "eval" / "qrels.jsonl")
qr = pd.DataFrame(qrels)
qr.groupby(["level", qr["index"].fillna("-"), "grade"]).size().unstack("grade")

Relevant (grade-2) units per question: with a 50-token overlap a short quote can lie in two consecutive windows, so a window question can have two grade-2 windows.

In [ ]:
g2 = qr[(qr["level"] == "unit") & (qr["grade"] == 2)].groupby(["index", "question_id"]).size()
g2.groupby("index").describe()[["mean", "min", "max"]]

## One example: a gold quote and the units that contain it

In [ ]:
q = next(q for q in questions if q["answerable"] and corpus.sections[q["_gold"][0]["section_id"]]["kind"] == "amc"
         and len(corpus.units_of(q["_gold"][0]["section_id"], "win500")) > 2)
g = q["_gold"][0]
print(q["question_id"], q["question"], "\n", g)
for index in corpus.indexes():
    for u in corpus.units_of(g["section_id"], index):
        inside = u["char_start"] <= g["char_start"] and u["char_end"] >= g["char_end"]
        print(f"  {index:6} {u['unit_id']:24} [{u['char_start']:6}, {u['char_end']:6})  {'contains the quote' if inside else ''}")